# AASTU Mini Semantic Search

**Week 2: Data, APIs & Embeddings** | AI/ML AWS AASTU Cohort 1

This notebook builds a search engine that finds FAQs by **meaning**, not by matching keywords.

**Pipeline:** load a messy CSV → clean it with functions → process numbers with NumPy → prepare the text → embed it → store it in ChromaDB → ask a question and get the top 3 matches.

> The FAQ data is **sample data for learning**, not official university policy.

## 1. Setup

This cell makes the notebook run in **two places**:
- **Colab:** it clones the public GitHub repo and installs `requirements.txt`.
- **Locally (VS Code / Jupyter):** it looks for the `src` folder (the one containing `load_data.py`) in the current folder, its parents and its children.

Then it adds `src` to Python's path so we can import our own modules.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_NAME = "week-02-aastu-semantic-search"
REPO_URL = f"https://github.com/kedirzeynu/{REPO_NAME}.git"

if "google.colab" in sys.modules:  # running in Colab
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_NAME)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )


def find_src():
    """Find the folder that holds load_data.py (works from any starting folder)."""
    here = Path.cwd()
    for base in [here, *here.parents]:
        for candidate in (base / "src", base / "week-02" / "src"):
            if (candidate / "load_data.py").exists():
                return candidate
    for candidate in [*here.glob("*/src"), *here.glob("*/week-02/src")]:
        if (candidate / "load_data.py").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find src/load_data.py starting from {here}. "
        "Open the project folder in VS Code and check the file names."
    )


SRC = find_src()
sys.path.insert(0, str(SRC))
print("Source folder:", SRC)
print("Modules found:", sorted(p.stem for p in SRC.glob("*.py")))

Source folder: /home/kedir/week-02-aastu-semantic-search/src
Modules found: ['cleaning', 'embeddings', 'load_data', 'numpy_stats', 'search', 'similarity', 'text_prep']


## 2. Imports

Libraries first (`numpy`, `pandas`), then **our own modules** from `src`. Each module does one job in the pipeline.

| Module | Job |
|---|---|
| `load_data` | load the raw CSV and take a first look |
| `cleaning` | the cleaning functions |
| `numpy_stats` | statistics with NumPy |
| `text_prep` | clean text, tokenize, chunk |
| `embeddings` | text → vectors |
| `similarity` | cosine similarity and nearest neighbors |
| `search` | ChromaDB storage and search |

In [4]:
import numpy as np
import pandas as pd

from load_data import load_faqs, first_look
from cleaning import clean_faqs, quality_report
from numpy_stats import summarize
from text_prep import prepare_text_columns, chunk_words
from embeddings import get_embedder, embed_texts
from similarity import cosine, top_k
from search import build_collection, search, print_hits, numpy_search

print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.5.4 | pandas 3.1.0


## 3. Load the raw dataset

`faq_raw.csv` is **deliberately messy**. We load it exactly as it is and run the **five first-look commands**:

1. `shape`: how big is it?
2. `head()`: what do the first rows look like?
3. `info()`: what are the column types and non-blank counts?
4. `describe()`: statistics for numeric columns
5. `isna().sum()`: how many blanks per column?

**What to look for:** `views` is stored as **text** (values like `"1,200"` contain a comma), so `describe()` only shows `id`. There are also blanks in `answer`, `category` and `views`.

In [5]:
raw = load_faqs()
first_look(raw)

SHAPE: (24, 6)

HEAD:
    id                                 question  \
0   1           How do I register for courses?   
1   2       When is the registration deadline?   
2   3  I forgot my password. What should I do?   
3   4                How can I reset my login?   
4   5                    Where is the library?   

                                              answer       category  views  \
0  Log in to the student portal, open the Registr...   Registration  1,200   
1  Registration closes at the end of the second w...  registration     950   
2  Use the Forgot Password link on the portal log...       Accounts   2300   
3  Click Forgot Password on the login page. A res...       ACCOUNTS  1,850   
4  The main library is next to the administration...         Campus    640   

   date_added  
0  2026-09-01  
1  2026-09-01  
2  2026-09-03  
3  2026-09-03  
4  2026-09-05  

INFO:
<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 6 columns):
 #   Column  

## 4. Clean the data with functions

`clean_faqs()` runs five cleaning functions **in a deliberate order**:

1. `fix_text`: trim spaces, tidy category labels (`ACCOUNTS` → `Accounts`, but `IT` stays `IT`)
2. `fix_types`: `"1,200"` → `1200.0`, date text → real dates
3. `remove_duplicates`: keep each question + answer once. This runs **after** text cleaning, because two rows that differ only by extra spaces become identical only after trimming.
4. `handle_outliers`: values outside the IQR range become blank
5. `handle_missing`: drop rows with no answer, fill `views` with the **median**, label missing categories `Unknown`

The rows lost are the exact duplicate, the near-duplicate and the row with no answer.

In [6]:
quality_report(raw, "BEFORE")
faqs = clean_faqs(raw)
quality_report(faqs, "AFTER")

faqs[["id", "category", "views", "date_added", "views_scaled"]].head(8)

--- BEFORE ---
rows: 24 | duplicate rows: 1
blanks: {'id': 0, 'question': 0, 'answer': 1, 'category': 1, 'views': 2, 'date_added': 0}
views: normal range -1587.5 to 3992.5, outliers found: 2
--- AFTER ---
rows: 21 | duplicate rows: 0
blanks: {'id': 0, 'question': 0, 'answer': 0, 'category': 0, 'views': 0, 'date_added': 1, 'views_scaled': 0, 'text': 0}


,id,category,views,date_added,views_scaled
0,1,Registration,1200.0,2026-09-01,0.341421
1,2,Registration,950.0,2026-09-01,0.254766
2,3,Accounts,2300.0,2026-09-03,0.722704
3,4,Accounts,1850.0,2026-09-03,0.566724
4,5,Campus,640.0,2026-09-05,0.147314
5,7,Fees,3100.0,2026-09-07,1.000000
6,8,Fees,880.0,2026-09-07,0.230503
7,9,Housing,780.0,2026-09-10,0.195841


## 5. Statistics with NumPy

`summarize()` uses NumPy to calculate the **count, mean, median, standard deviation, min, max and the 25th/75th percentiles** of the cleaned `views`.

Compare the **mean** and the **median**. The median is not pulled around by extreme values, which is why we used it to fill blanks. We also scaled views to the range 0 to 1 (`views_scaled`) so numbers on different scales can be compared fairly.

In [7]:
stats = summarize(faqs["views"])
print(pd.Series(stats).round(2))

print("\nviews_scaled range:",
      faqs["views_scaled"].min(), "to", faqs["views_scaled"].max())

count       21.00
mean      1065.48
median     880.00
std        709.08
min        215.00
max       3100.00
p25        640.00
p75       1200.00
dtype: float64

views_scaled range: 0.0 to 1.0


## 6. Prepare the text and chunk it

Text needs its own cleaning, separate from the table cleaning:

- `clean_text`: lowercase, remove symbols, collapse spaces
- `tokenize`: split into word tokens
- `n_tokens`: how long each FAQ is

**Chunking** cuts long text into overlapping pieces (about 200 words plus overlap is a good size for a real handbook). Our FAQs are only about 16 to 23 tokens each, so **each FAQ is already chunk-sized**, and we embed one FAQ per chunk. To prove the chunker works, we join all the answers into a mini "handbook" and chunk it with `size=60`, `overlap=12`.

In [8]:
faqs = prepare_text_columns(faqs)
print(faqs[["id", "n_tokens", "text_clean"]].head(3).to_string())

tokens = faqs["n_tokens"].to_numpy()
print("\ntokens per FAQ: mean", tokens.mean().round(1),
      "| median", np.median(tokens),
      "| min", tokens.min(), "| max", tokens.max())

handbook_words = " ".join(faqs["answer"]).split()
chunks = chunk_words(handbook_words, size=60, overlap=12)
print("\nhandbook words:", len(handbook_words))
print("chunk sizes:", [len(c) for c in chunks])
print("overlap works:", chunks[0][-12:] == chunks[1][:12])

   id  n_tokens                                                                                                                        text_clean
0   1        23  how do i register for courses log in to the student portal open the registration tab and select your courses before the deadline
1   2        17                               when is the registration deadline registration closes at the end of the second week of the semester
2   3        23     i forgot my password what should i do use the forgot password link on the portal login page and follow the email instructions

tokens per FAQ: mean 19.6 | median 19.0 | min 16 | max 23

handbook words: 283
chunk sizes: [60, 60, 60, 60, 60, 43]
overlap works: True


## 7. Embeddings: meaning as numbers

An **embedding** is a vector of numbers describing one text. Texts with similar meaning land **near each other**. We use the `all-MiniLM-L6-v2` model, which produces **384 numbers** per text.

We call `get_embedder(allow_fallback=False)` so that if the real model can't download, the notebook **stops with an error** instead of quietly using the weak offline stand-in. This guarantees the saved outputs come from the real model.

> The first run downloads about 80 MB, so it needs internet.

In [10]:
embedder = get_embedder(allow_fallback=False)
print("Embedder:", embedder.name)

vectors = embed_texts(embedder, faqs["text"])
print("vectors shape:", vectors.shape)
assert vectors.shape == (len(faqs), 384), "unexpected embedding shape"

print("\nfirst 5 numbers of the first FAQ:", vectors[0][:5].round(3))

/home/kedir/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 135k/79.3M [00:19<3:20:05, 6.92kiB/s] 


ReadTimeout: The read operation timed out

## 8. Cosine similarity: comparing directions

To measure how close two vectors are, we compare their **direction**:

- **1** means the same direction (same meaning)
- **0** means unrelated
- **-1** means opposite

`cosine(a, b) = a @ b / (norm(a) * norm(b))`. Dividing by the lengths means only direction matters, not length.

Below, we compare "I forgot my password" with a similar sentence that shares **no important words**, and with an unrelated sentence. If the model works, the first score should be clearly higher than the second.

In [ ]:
sentences = [
    "I forgot my password",
    "I can't log in to my account",
    "Where do I find the injera recipe?",
]
a, b, c = embed_texts(embedder, sentences)

print(f"forgot password  vs  can't log in : {cosine(a, b):.2f}")
print(f"forgot password  vs  injera recipe: {cosine(a, c):.2f}")

## 9. Nearest neighbors with NumPy only

Now we search **by hand**, with no database:

1. Embed the question into a vector `q`.
2. Score **every** FAQ at once (`docs @ q`, divided by the lengths).
3. `np.argsort(scores)[::-1][:k]` gives the positions of the **k best** scores. `argsort` sorts ascending, `[::-1]` reverses it, and `[:k]` keeps the top k.

With **k = 3** we return the 3 closest FAQs. This is **brute force**: it compares with every vector, which is fine for about 20 FAQs but slow at millions. That is why vector databases use indexes.

In [ ]:
query = "I forgot my password"
q = embed_texts(embedder, [query])[0]

idx, scores = top_k(vectors, q, k=3)

print(f'NumPy top 3 for: "{query}"\n')
for rank, (i, s) in enumerate(zip(idx, scores), start=1):
    print(f"{rank}. [{s:.2f}] {faqs.iloc[i]['question']}")

print("\nSame search through numpy_search (id, score):")
print(numpy_search(faqs, embedder, query, k=3))

## 10. Store in ChromaDB and ask questions

A **vector database** stores vectors and finds the nearest ones quickly, using an index instead of brute force (Chroma uses HNSW, a graph of links that hops closer step by step).

What `build_collection` does:
- Stores each FAQ's **text** and **vector**.
- Stores **category** and **views** alongside each vector, so we can filter later.
- Uses **cosine** distance. Chroma's default is not cosine, so we set it on purpose.

Search results show `score = 1 - distance`, so higher is better. We ask three questions and print the **top 3 matches** for each.

In [ ]:
col = build_collection(faqs, embedder)
print("FAQs stored in Chroma:", col.count())

for question in [
    "I forgot my password",
    "When is registration?",
    "Where can I study quietly?",
]:
    print_hits(question, search(col, embedder, question, k=3))

## 11. Similarity + filters

Because we stored fields next to each vector, we can combine **meaning** with **conditions**: "closest to this question, but only among Fees FAQs", or "only popular ones".

- `where={"category": "Fees"}` filters by text.
- `where={"views": {"$gt": 1000}}` filters by number.

In [ ]:
query = "how do I pay"

print("No filter:")
print_hits(query, search(col, embedder, query, k=3))

print("\nOnly the Fees category:")
print_hits(query, search(col, embedder, query, k=3, where={"category": "Fees"}))

print("\nOnly FAQs with more than 1000 views:")
print_hits(query, search(col, embedder, query, k=3, where={"views": {"$gt": 1000}}))

## 12. Try your own question

Change the text below to anything a student might ask. Try questions with **different words** from the FAQ, for example "how do I get back into my account" or "can I get a copy of my grades". Semantic search should still find the right FAQ.

In [ ]:
my_question = "how do I get back into my account"

print_hits(my_question, search(col, embedder, my_question, k=3))

## 13. What I learned

*(Write 3 to 4 sentences in your own words. Replace these prompts.)*

- Which cleaning problem surprised me most, and what would have gone wrong without fixing it?
- What did semantic search find that a keyword search would have missed? (Give one example from my queries.)
- Why do vector databases use an index instead of comparing with every vector?
- What would I improve next, for example more FAQs, a larger model, or a web page for the search?